# DKI on Google Colab

End-to-end run of the refactored DKI (Data-driven Keystone
Identification) framework. This notebook:

1. Clones the repo and installs deps.
2. Lets you use the bundled gLV data or upload your own abundance CSV
   (samples-as-rows **or** taxa-as-rows).
3. Trains the Phase-1 model (batched `dopri5` replicator ODE, cosine
   LR, early stopping, best-val checkpoint).
4. Predicts on the test set.
5. Computes **classical structural keystoneness** for every
   (sample, species) pair (Python port of `Keystoneness_computing.R`).
6. *(Optional)* Trains the **original DKI model** for head-to-head
   comparison.

Tip: switch the runtime to GPU (Runtime → Change runtime type → T4)
for a speedup on larger N — the trainer auto-detects CUDA / MPS / CPU.

## Research roadmap

The `dki/` package now implements **all six phases**. This notebook runs the
**Phase-1** pipeline by default; Phases 2, 3 and 6 are opt-in via
`TrainConfig` / CLI flags, and Phases 4-5 are extra analysis steps
(`dki.ensemble`, `dki.keystoneness.null_model_keystoneness`,
`dki.extensions.shapley`).

| Phase | Status | What it adds |
|---|---|---|
| **1. Faithful refactor + batched integration** | ✅ landed | `dki/` package, batched `dopri5` replicator ODE (`rtol=1e-5`, `atol=1e-7`, t=[0, 100]), cosine LR, gradient clipping at 1.0, early stop on val BC, best-val checkpoint, auto CUDA/MPS/CPU. ~600× per-epoch speedup over the original. |
| **2. Nonlinear ODEFunc + composite loss** | ✅ landed | `--nonlinear` → fitness `fc2(SiLU(fc1(y)))` with hidden dim `hidden_mult·N` (default `2N`); the two-stacked-`Linear` cNODE2 is provably equivalent to a single `Linear` (W2·W1 = W) and `tests/test_phase2.py` proves it. `--loss composite --alpha 0.3` adds `α·BC + (1−α)·CLR-MSE` for rare-species accuracy. |
| **3. Deep-equilibrium reformulation** | ✅ landed | `--mode deq`: solves the replicator fixed point with **safeguarded** Anderson acceleration (50 iters, tol 1e-6) on a simplex-preserving mirror map, backprop via the implicit function theorem, ODE fallback on non-convergence. Matches the ODE within BC < 0.02 on the potential-game test. |
| **4. Ensembles + uncertainty + null-model normalisation** | ✅ landed | `dki.ensemble.train_ensemble` fits K bootstrap models (default K=5) → `EnsemblePredictor` returns `(mean, std)`. `null_model_keystoneness` adds an **alternative** z-score calibration (up to 50 abundance-matched null species) alongside — not replacing — the classical `(1−p)` formula. |
| **5. Shapley keystoneness** *(extension)* | ✅ landed | `dki.extensions.shapley`: Monte-Carlo Shapley (default `n_perm=200`) for a **different question** — synergy/redundancy-aware contribution. Reported as `k_shapley_synergistic`, never replacing `k_classical` or `k_zscore`. |
| **6. Self-consistency regulariser** | ✅ landed | `--consistency-weight 0.1`: mask one present species, predict `q'`, re-feed `q'` and require the re-integrated prediction matches `q'` under BC. Pushes predictions to be genuine fixed points (tighter ensemble std). |

Throughout: the original `DKI.py` is preserved at
`legacy/DKI_original.py`, `Keystoneness_computing.R` stays runnable as
a cross-check, and `pytest` covers simplex preservation, loss
correctness, batched-vs-loop equivalence, the keystoneness port, and a
dedicated `tests/test_phase{2..6}.py` for each new phase.

Design notes pinned by the project:
* The metacommunity assumption (same `f` across all samples, only `z`
  varies) is preserved — the ODEFunc takes **only `y`**; no covariate
  conditioning, hypernetworks, or context-dependent interactions.
* Classical Paine-style keystoneness stays in the default output; null
  z-score and Shapley land as alternatives, not replacements.

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/metagenAu/DKI.git'
BRANCH   = 'claude/peaceful-goodall-4AC8l'   # change to 'main' once merged
REPO_DIR = '/content/DKI'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
%cd $REPO_DIR
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

import torch, numpy as np
from dki.device import auto_device
print('torch', torch.__version__, 'device', auto_device())

## 2. Pick your data

**Option A** — use the bundled gLV synthetic data (works out of the box).

**Option B** — upload your own CSV (numeric abundance table). The cell
below handles both orientations:
* `samples_as_rows=True`  → rows are samples, columns are taxa
* `samples_as_rows=False` → rows are taxa,   columns are samples (legacy)

Counts or relative abundances both work — each sample is renormalised to
sum to 1 internally.

In [ ]:
USE_BUNDLED = True              # set False to upload your own
samples_as_rows = True          # only matters when USE_BUNDLED=False
header_row = False              # set True if your CSV has a header row
index_col  = False              # set True if your CSV has a row-label column

DATA_DIR = '/content/DKI/data' if USE_BUNDLED else '/content/dki_data'

if not USE_BUNDLED:
    from google.colab import files
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Upload your abundance CSV (and optionally a test CSV).')
    uploaded = files.upload()
    import pandas as pd
    for name, _ in uploaded.items():
        df = pd.read_csv(name,
                         header=0 if header_row else None,
                         index_col=0 if index_col else None)
        arr = df.to_numpy(dtype=np.float32)
        if samples_as_rows:
            arr = arr.T    # -> (n_taxa, n_samples) for the DKI loader
        dst = os.path.join(DATA_DIR, 'Ptrain.csv' if 'train' in name.lower() or len(uploaded)==1
                                       else 'Ptest.csv')
        np.savetxt(dst, arr, delimiter=',')
        print(f'  wrote {dst}  shape={arr.shape}  (taxa, samples)')

print('Data dir:', DATA_DIR)
!ls -la $DATA_DIR

## 3. Train the new model

In [ ]:
from dki.train import TrainConfig, train

cfg = TrainConfig(
    data_dir=DATA_DIR,
    out_dir='/content/results',
    epochs=1000,              # matches the original DKI.py budget; one
    batch_size=20,            # minibatch step per epoch -> ~50 passes
    lr=1e-2,                  # over a 400-sample train set.
    min_lr=1e-4,
    t_final=100.0,
    grad_clip=1.0,
    early_stop_patience=200,  # val BC is noisy; allow long plateaus.
    val_fraction=0.2,
    seed=0,
    save_predictions=True,
)
model, result, data = train(cfg)
print(f'\nBest val BC: {result.best_val_loss:.6f} at epoch {result.best_epoch}')
print(f'Mean epoch wall-clock: {np.mean(result.epoch_seconds):.3f}s')
print(f'Total wall-clock: {np.sum(result.epoch_seconds):.1f}s')

## 4. Loss curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(result.train_loss, label='train BC')
ax.plot(result.val_loss,   label='val BC')
ax.set_xlabel('epoch'); ax.set_ylabel('Bray-Curtis')
ax.set_title('DKI Phase-1 training')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Predict on test set

In [ ]:
from dki.infer import predict
from dki.losses import bray_curtis

if data.p_test is not None and data.z_test is not None:
    qtst = predict(model, data.z_test, t_final=cfg.t_final).cpu().numpy()
    qtrn = predict(model, data.z_all,  t_final=cfg.t_final).cpu().numpy()
    test_bc = bray_curtis(torch.from_numpy(qtst), data.p_test.cpu()).item()
    print(f'Test Bray-Curtis (mean over {qtst.shape[0]} samples): {test_bc:.4f}')
else:
    print('No test set found in', DATA_DIR)
    qtst = qtrn = None

## 6. Keystoneness

Computes the **classical structural keystoneness** from Wang et al.
(bioRxiv 2023.03.15.532858v3) for every (sample, species) pair listed
in `Sample_id.csv` / `Species_id.csv`:

$$
k_{\text{classical}}(i, s) \;=\; \mathrm{BC}\!\left(\tilde q_{i,\setminus s},\; q_{i,\setminus s}\right) \;\cdot\; (1 - p_{s,i})
$$

where $\tilde q_{i,\setminus s}$ is the full-community prediction with
species $s$ masked-and-renormalised, and $q_{i,\setminus s}$ is the
model's leave-one-out prediction.

Phase 4 (planned) will add an alternative null-model z-score calibration;
Phase 5 (planned) will add a synergy-aware Shapley extension. Both will
be reported alongside `k_classical`, not replacing it.

In [ ]:
import pandas as pd
from dki.keystoneness import classical_structural_keystoneness

sample_id  = np.loadtxt(os.path.join(DATA_DIR, 'Sample_id.csv'),  delimiter=',').astype(int)
species_id = np.loadtxt(os.path.join(DATA_DIR, 'Species_id.csv'), delimiter=',').astype(int)

# Legacy on-disk orientation: (n_species, n_samples_or_pairs).
Ptrain = np.loadtxt(os.path.join(DATA_DIR, 'Ptrain.csv'), delimiter=',')
Ptest  = np.loadtxt(os.path.join(DATA_DIR, 'Ptest.csv'),  delimiter=',')
Ptrain = Ptrain / Ptrain.sum(axis=0, keepdims=True).clip(min=1e-12)
Ptest  = Ptest  / Ptest.sum(axis=0, keepdims=True).clip(min=1e-12)

kdf = classical_structural_keystoneness(
    qtrn=qtrn, qtst=qtst, ptrn=Ptrain, ptst=Ptest,
    sample_id=sample_id, species_id=species_id,
)
kdf.head()

In [ ]:
from scipy.stats import spearmanr

rho, pval = spearmanr(kdf['k_true'], kdf['k_pred'])
print(f'Spearman(k_true, k_pred) = {rho:.3f}  (n={len(kdf)}, p={pval:.2g})')

fig, ax = plt.subplots(figsize=(5,5))
ax.hexbin(kdf['k_true'], kdf['k_pred'], gridsize=40, mincnt=1, cmap='Spectral_r')
lim = max(kdf['k_true'].max(), kdf['k_pred'].max())
ax.plot([0, lim], [0, lim], color='#d01c8b', lw=1)
ax.set_xlabel(r'$k_\mathrm{classical}$ (true)')
ax.set_ylabel(r'$k_\mathrm{classical}$ (predicted)')
ax.set_title(f'Structural keystoneness   Spearman $\\rho$={rho:.2f}')
plt.show()

kdf.to_csv('/content/results/keystoneness.csv', index=False)
print('Saved /content/results/keystoneness.csv')

In [ ]:
# Top-10 predicted keystone species (highest k_pred)
kdf.sort_values('k_pred', ascending=False).head(10)

## 6b. Assessing the keystoneness fit

Global Spearman $\rho$ mixes two separate questions:

1. **Does the base model fit communities?** — full-community prediction vs observed.
2. **Does it extrapolate to leave-one-out perturbations?** — the LOO prediction the keystoneness formula actually depends on.

The first cell below splits the Bray-Curtis error between these two regimes
(if the base fit is good but LOO is poor, the keystoneness error is an
*extrapolation* problem). The second adds **top-k overlap / precision@k** and a
**per-species** view — what matters when you only care about the strongest
keystones, which a single global $\rho$ can hide.

In [ ]:
# --- Localise the error: full-community fit vs leave-one-out (perturbation) fit ---
# Predictions are (n, n_species); observed Ptrain/Ptest are (n_species, n) (legacy).
def per_col_bc(pred, obs):
    o = obs.T                                   # -> (n, n_species)
    num = np.abs(pred - o).sum(axis=1)
    den = np.clip(np.abs(pred + o).sum(axis=1), 1e-12, None)
    return num / den

bc_full = per_col_bc(qtrn, Ptrain)              # full-community samples
bc_loo  = per_col_bc(qtst, Ptest)               # leave-one-out assemblages

print(f'Full-community BC : mean {bc_full.mean():.4f}  median {np.median(bc_full):.4f}  (n={len(bc_full)})')
print(f'Leave-one-out  BC : mean {bc_loo.mean():.4f}  median {np.median(bc_loo):.4f}  (n={len(bc_loo)})')
print(f'(README Phase-3 target: mean BC < 0.01)')

fig, ax = plt.subplots(figsize=(6,4))
bins = np.linspace(0, max(bc_full.max(), bc_loo.max()), 41)
ax.hist(bc_full, bins=bins, alpha=0.6, density=True, label=f'full-community (mean {bc_full.mean():.3f})')
ax.hist(bc_loo,  bins=bins, alpha=0.6, density=True, label=f'leave-one-out (mean {bc_loo.mean():.3f})')
ax.set_xlabel('Bray-Curtis (predicted vs observed)'); ax.set_ylabel('density')
ax.set_title('Where the error lives: base fit vs perturbation extrapolation')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
# --- Ranking quality: what matters when you only care about the top keystones ---
def precision_at_k(true_vals, pred_vals, ks):
    order_t = np.argsort(-true_vals)
    order_p = np.argsort(-pred_vals)
    out = {}
    for k in ks:
        kk = min(k, len(true_vals))
        out[k] = len(set(order_t[:kk]) & set(order_p[:kk])) / kk
    return out

ks = [5, 10, 20, 50, 100]

# (a) Pair-level — every (sample, species) pair (directly comparable to global rho).
pk_pair = precision_at_k(kdf['k_true'].to_numpy(), kdf['k_pred'].to_numpy(), ks)

# (b) Species-level — aggregate to one keystoneness per species (mean over samples):
#     the practical "which species are keystones?" question.
sp = kdf.groupby('species').agg(k_true=('k_true', 'mean'), k_pred=('k_pred', 'mean'))
rho_sp, _ = spearmanr(sp['k_true'], sp['k_pred'])
ks_sp = [k for k in ks if k <= len(sp)]
pk_sp = precision_at_k(sp['k_true'].to_numpy(), sp['k_pred'].to_numpy(), ks_sp)

print('Top-k overlap (fraction of the true top-k recovered in the predicted top-k)')
print(f'{"k":>5} | {"pair-level":>11} | {"species-level":>13}')
for k in ks:
    pp = f'{pk_pair[k]:.2f}'
    ss = f'{pk_sp[k]:.2f}' if k in pk_sp else '-'
    print(f'{k:>5} | {pp:>11} | {ss:>13}')
print(f'\nSpecies-level Spearman = {rho_sp:.3f}  (n_species={len(sp)})')

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(sp['k_true'], sp['k_pred'], s=18, alpha=0.7)
lim = max(sp['k_true'].max(), sp['k_pred'].max())
ax.plot([0, lim], [0, lim], color='#d01c8b', lw=1)
ax.set_xlabel(r'$k_\mathrm{classical}$ (true, per-species mean)')
ax.set_ylabel(r'$k_\mathrm{classical}$ (predicted, per-species mean)')
ax.set_title(f'Per-species keystoneness   Spearman $\\rho$={rho_sp:.2f}')
plt.show()

## 7. (Optional) Compare to the original DKI model

Trains the legacy 2-Linear ODEFunc on the same split, same seed, using
the unchanged training loop (per-sample for-loop, 10,000-step Euler
integration). On a Colab CPU this is **~170 s / epoch** for the gLV
data, so the default below is 3 epochs (~9 minutes).

The new model on the same 3-5 epochs already matches or beats the
legacy val BC, and is ~600× faster per epoch.

Set `RUN_LEGACY = True` to enable.

In [ ]:
RUN_LEGACY = False
LEGACY_EPOCHS = 3

if RUN_LEGACY:
    out = subprocess.run([
        sys.executable, 'legacy/baseline_runner.py',
        '--data', DATA_DIR,
        '--epochs', str(LEGACY_EPOCHS),
        '--seed', str(cfg.seed),
        '--val-fraction', str(cfg.val_fraction),
        '--out', '/content/results/legacy',
    ], capture_output=True, text=True)
    print(out.stdout)
    if out.returncode != 0:
        print('STDERR:', out.stderr)
else:
    print('Skipping legacy run. Flip RUN_LEGACY = True to enable.')

In [ ]:
# Plot val BC: new (200 ep) vs legacy (LEGACY_EPOCHS ep).
if RUN_LEGACY and os.path.exists('/content/results/legacy/val_loss.npy'):
    legacy_val = np.load('/content/results/legacy/val_loss.npy')
    legacy_t   = np.load('/content/results/legacy/epoch_times.npy')
    new_val    = np.array(result.val_loss)
    new_t      = np.array(result.epoch_seconds)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(new_val,    label=f'new  (mean {new_t.mean():.2f}s/ep)')
    axes[0].plot(legacy_val, label=f'legacy (mean {legacy_t.mean():.1f}s/ep)', marker='o')
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('val BC')
    axes[0].set_title('Val Bray-Curtis vs epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(np.cumsum(new_t),    new_val,    label='new')
    axes[1].plot(np.cumsum(legacy_t), legacy_val, label='legacy', marker='o')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('cumulative wall-clock (s, log)')
    axes[1].set_ylabel('val BC')
    axes[1].set_title('Val Bray-Curtis vs compute'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    speedup = legacy_t.mean() / new_t.mean()
    print(f'Per-epoch speedup: {speedup:.0f}x  '
          f'(legacy {legacy_t.mean():.1f}s vs new {new_t.mean():.2f}s)')
    print(f'Best val BC — new: {min(new_val):.4f}    legacy: {min(legacy_val):.4f}')
else:
    print('Legacy comparison disabled or not yet run.')

## 8. Phases 2–6 in action

Sections 1–7 run the **Phase-1** baseline (linear fitness, BC loss, ODE
integration). The cells below exercise each later phase on the *same* data and
split, so the numbers compare directly against the Phase-1 run above.

* **Phase 2** — nonlinear SiLU fitness + composite BC/CLR loss
* **Phase 3** — deep-equilibrium fixed-point solver (`mode='deq'`)
* **Phase 6** — self-consistency regulariser
* **Phase 4** — K=5 ensemble (uncertainty) + null-model z-score keystoneness
* **Phase 5** — Monte-Carlo Shapley keystoneness (synergy-aware)

Phases 4–5 train an ensemble and run many perturbation predictions, so they sit
behind a `RUN_HEAVY` flag. `DEMO_EPOCHS` defaults to the full Phase-1 budget
(`cfg.epochs`) so every phase is directly comparable to the baseline; lower it for a
quick smoke pass.

In [ ]:
RUN_HEAVY   = False    # Phase 4 (ensemble) + Phase 5 (Shapley): set True for the full demo
DEMO_EPOCHS = cfg.epochs   # match the Phase-1 budget so phase numbers are directly comparable
                           # (Phase 4/5 train a K=5 ensemble = 5 full runs; lower this if too slow)
phase_bc = {'1 baseline': result.best_val_loss}   # collect best val BC per phase for the summary

### Phase 2 — nonlinear SiLU fitness + composite loss

`fc2(SiLU(fc1(y)))` makes the per-capita fitness genuinely nonlinear (the legacy
two-`Linear` stack collapses to a single affine map), and the objective becomes
`α·BC + (1−α)·CLR-MSE` (α=0.3) to sharpen rare-species accuracy.

In [ ]:
cfg2 = TrainConfig(data_dir=DATA_DIR, out_dir='/content/results/phase2',
                   epochs=DEMO_EPOCHS, nonlinear=True, loss='composite', alpha=0.3,
                   seed=cfg.seed, val_fraction=cfg.val_fraction)
model2, result2, _ = train(cfg2, data=data)
phase_bc['2 nonlinear+composite'] = result2.best_val_loss
print(f"Phase-2 best val BC: {result2.best_val_loss:.4f}   "
      f"(Phase-1 baseline: {result.best_val_loss:.4f})")

### Phase 3 — deep-equilibrium solver (`mode='deq'`)

Solves the replicator fixed point directly with safeguarded Anderson
acceleration (backprop via the implicit function theorem) instead of integrating
the ODE. The check below confirms the DEQ and ODE solvers agree on the *same*
weights (README target: mean BC < 0.01).

In [ ]:
cfg3 = TrainConfig(data_dir=DATA_DIR, out_dir='/content/results/phase3',
                   epochs=DEMO_EPOCHS, mode='deq',
                   seed=cfg.seed, val_fraction=cfg.val_fraction)
model3, result3, _ = train(cfg3, data=data)
phase_bc['3 deq'] = result3.best_val_loss

q_ode = predict(model3, data.z_val, mode='ode')
q_deq = predict(model3, data.z_val, mode='deq')
agree = bray_curtis(q_ode, q_deq).item()
print(f"Phase-3 (DEQ) best val BC: {result3.best_val_loss:.4f}")
print(f"DEQ vs ODE agreement on val (same weights): mean BC = {agree:.4f}  (target < 0.01)")

### Phase 6 — self-consistency regulariser

Auxiliary loss: mask one present species, predict the reduced community's
equilibrium `q'`, re-feed it, and require the second prediction to match `q'`.
This pushes predictions toward genuine fixed points (steadier keystoneness under
perturbation).

In [ ]:
cfg6 = TrainConfig(data_dir=DATA_DIR, out_dir='/content/results/phase6',
                   epochs=DEMO_EPOCHS, consistency_weight=0.1,
                   seed=cfg.seed, val_fraction=cfg.val_fraction)
model6, result6, _ = train(cfg6, data=data)
phase_bc['6 self-consistency'] = result6.best_val_loss
print(f"Phase-6 best val BC: {result6.best_val_loss:.4f}   "
      f"(Phase-1 baseline: {result.best_val_loss:.4f})")

### Phase 4 — K=5 ensemble + null-model z-score keystoneness

Trains K bootstrap-resampled models so predictions carry a `(mean, std)`
uncertainty, and recalibrates keystoneness as a **z-score** against
abundance-matched null species in the same community — reported *alongside*
`k_classical`, not replacing it. Runs on a random subset of pairs for speed.

In [ ]:
from dki.ensemble import train_ensemble, EnsemblePredictor
from dki.keystoneness import null_model_keystoneness

ens = None
if RUN_HEAVY:
    ecfg = TrainConfig(data_dir=DATA_DIR, out_dir='/content/results/ensemble',
                       epochs=DEMO_EPOCHS, seed=cfg.seed, val_fraction=cfg.val_fraction)
    models, _, edata = train_ensemble(ecfg, k=5, data=data)
    ens = EnsemblePredictor(models)
    _, std = ens.predict(edata.z_all)
    print(f"Ensemble predictive std over {len(models)} members: "
          f"mean {std.mean():.4f}  max {std.max():.4f}")

    rng_demo = np.random.default_rng(0)
    pick = rng_demo.choice(len(sample_id), size=min(300, len(sample_id)), replace=False)
    df_z = null_model_keystoneness(ens.mean_predict_fn(), edata.z_all, Ptrain,
                                   sample_id[pick], species_id[pick], n_null=50)
    print(df_z.sort_values('k_zscore', ascending=False).head(10).to_string(index=False))
else:
    print("Set RUN_HEAVY = True to train the K=5 ensemble and compute z-score keystoneness.")

### Phase 5 — Monte-Carlo Shapley keystoneness

A *different* question from the classical definition: each species' average
marginal contribution across random removal orders (synergy/redundancy-aware),
reported as `k_shapley_synergistic`. Uses the ensemble-mean predictor on a small
subset of pairs with reduced `n_perm`; raise `n_perm` toward 200 for the paper
setting.

In [ ]:
from dki.extensions.shapley import shapley_keystoneness

if RUN_HEAVY and ens is not None:
    rng_demo = np.random.default_rng(1)
    pick = rng_demo.choice(len(sample_id), size=min(60, len(sample_id)), replace=False)
    df_s = shapley_keystoneness(ens.mean_predict_fn(), data.z_all,
                                sample_id[pick], species_id[pick], n_perm=50)
    print(df_s.sort_values('k_shapley_synergistic', ascending=False).head(10).to_string(index=False))
else:
    print("Set RUN_HEAVY = True (and run the Phase-4 cell first) to compute Shapley keystoneness.")

### Phase comparison — best validation Bray-Curtis

In [ ]:
summary = pd.DataFrame(
    [{'phase': k, 'best_val_BC': v} for k, v in phase_bc.items()]
).set_index('phase')
print(summary.to_string(float_format=lambda x: f'{x:.4f}'))

## 9. Save and download artifacts

In [ ]:
!ls -la /content/results
# Uncomment to download:
# from google.colab import files
# files.download('/content/results/best_model.pt')
# files.download('/content/results/qtst.csv')
# files.download('/content/results/qtrn.csv')
# files.download('/content/results/keystoneness.csv')

---

**Phases 2–6 are now implemented** in the `dki/` package. Quick reference:

```python
from dki.train import TrainConfig, train

# Phase 2 — nonlinear SiLU fitness + composite BC/CLR loss
cfg = TrainConfig(data_dir=DATA_DIR, nonlinear=True, loss='composite', alpha=0.3)

# Phase 3 — deep-equilibrium fixed-point solver
cfg = TrainConfig(data_dir=DATA_DIR, mode='deq')

# Phase 6 — self-consistency regulariser
cfg = TrainConfig(data_dir=DATA_DIR, consistency_weight=0.1)

model, result, data = train(cfg)
```

```python
# Phase 4 — K=5 ensemble + null-model z-score keystoneness
from dki.ensemble import train_ensemble, EnsemblePredictor
from dki.keystoneness import null_model_keystoneness

models, _, data = train_ensemble(TrainConfig(data_dir=DATA_DIR), k=5)
ens = EnsemblePredictor(models)            # ens.predict(z) -> (mean, std)
df_z = null_model_keystoneness(ens.mean_predict_fn(), data.z_all, Ptrain,
                               sample_id, species_id, n_null=50)

# Phase 5 — Monte-Carlo Shapley keystoneness (synergy-aware extension)
from dki.extensions.shapley import shapley_keystoneness
df_s = shapley_keystoneness(ens.mean_predict_fn(), data.z_all,
                            sample_id, species_id, n_perm=200)
```

The equivalent CLI flags: `python -m dki.train --data data --nonlinear
--loss composite --alpha 0.3 --mode deq --consistency-weight 0.1`.